In [19]:
import json
from pathlib import Path
from bs4 import BeautifulSoup
import re
from html import escape
from pathlib import Path
import html as ihtml

import re
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional


from neo4j import GraphDatabase, Transaction 

import networkx as nx
import matplotlib.pyplot as plt

from neo4j import GraphDatabase
import math, os

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"



In [2]:
PREFIX = "(document name and content will vary by organization):"

def slugify(name: str) -> str:
    """Deterministic, Neo4j-friendly ID from document name."""
    s = name.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s or "doc"

def expand_related_documents(raw_related: List[Dict]) -> List[Dict]:
    """
    Take the raw related_documents list and return normalized items:
    [
      {
        "subcontrol_id": 9292373,
        "doc_id": "asset_management_policy",
        "name": "Asset Management Policy",
        "raw": "(full original string)",
      },
      ...
    ]
    """
    out = []

    for r in raw_related:
        sid = r.get("subcontrol_id")
        raw = (r.get("description") 
               or r.get("id") 
               or "").strip()
        if not raw:
            continue

        text = raw
        # Strip standard prefix if present
        if text.lower().startswith(PREFIX.lower()):
            text = text[len(PREFIX):].strip()

        # Extract numbered bullets like "1) Something..."
        bullets = re.findall(
            r'\d+\)\s*(.+?)(?=(?:\n\s*\d+\)\s)|\Z)',
            text,
            flags=re.DOTALL
        )

        # Fallback: if no bullets found, treat entire text as one item
        if not bullets:
            bullets = [text]

        for b in bullets:
            name = " ".join(b.split())  # collapse whitespace/newlines
            if not name:
                continue
            doc_id = slugify(name)
            out.append(
                {
                    "subcontrol_id": sid,
                    "doc_id": doc_id,
                    "name": name,
                    "raw": raw,
                }
            )

    return out


def to_valid_db_name(framework_name: str) -> str:
    """
    Convert any framework name into a valid Neo4j database name.
    Rules:
      - lowercase only
      - only a-z, 0-9, dot, dash
      - must start with a letter
      - max length 63
    """
    # lowercase
    name = framework_name.lower()

    # replace invalid chars (anything not allowed) with dash
    name = re.sub(r'[^a-z0-9.-]', '-', name)

    # must start with a letter: if not, prepend "db-"
    if not re.match(r'^[a-z]', name):
        name = "db-" + name

    # truncate to 63 chars
    name = name[:63]

    return name


def extract_framework(data):
    framework = data.get('data').get('framework', [])
    extracted_data = []
    extracted_data.append({
        'id': framework.get('id'),
        'name': framework.get('name'),
        'description': framework.get('description')
    })
    
    return extracted_data

def extract_apps(data):
    apps = data.get('data').get('apps', [])
    extracted_data = []

    for app in apps:
        extracted_data.append({
            'name': app.get('name'),
            'description': app.get('description'),
            'id': app.get('id')
        })
    
    return extracted_data

def extract_sub_controls(data):
    sub_controls = data.get('data').get('sub_controls', [])
    extracted_data = []

    for sub_control in sub_controls:
        extracted_data.append({
            'id': sub_control.get('id'),
            'title': sub_control.get('title'),
            'description': sub_control.get('description'),
            'notes': sub_control.get('notes'),
            'policy_section': sub_control.get('policy_section'),
            'app_id': sub_control.get('app_id'),
            'identifier': sub_control.get('identifier'),
        })
    
    return extracted_data

def convert_html_content_to_list(html_content):
    
    soup = BeautifulSoup(html_content, "html.parser")
    
    # Prepare the base list
    structured_content = []

    # Extract sections by `sub-sub-head` class
    sections = soup.find_all("div", class_="sub-sub-head")
    for section in sections:
        section_title = section.get_text(strip=True).replace(":", "")  # Extract section title
        section_content = section.next_sibling

        content = {"title": section_title, "items": []}

        # Collect all subsequent sibling elements until the next section
        sibling = section
        while sibling and (sibling := sibling.next_sibling):
            if sibling.name == "div" and "sub-sub-head" in sibling.get("class", []):
                break  # Stop at the next section
            if sibling.string and sibling.string.strip():
                # Handle numbered items
                for line in sibling.string.split("<br/>"):
                    line = line.strip()
                    if line and line[0].isdigit():  # Remove leading numbers
                        item_text = line.split(")", 1)[1].strip() if ")" in line else line
                        content["items"].append(item_text)
                    else:
                        content["items"].append(line)

        try:
            if not content["items"] and section_content:
                text = str(section_content).strip()
                if text:
                    content["items"].append(text)

        except Exception as e:
            print(f"Error: {e}")
            print(f"Section: {section_title}")
            print(f"Content: {section_content}")
            print(f"Items: {content['items']}")

        structured_content.append(content)

    return structured_content

def html_to_text(html_snippet: str) -> str:
    soup = BeautifulSoup(html_snippet, "html.parser")
    # Turn <br> into newlines and strip extra whitespace
    txt = soup.get_text(separator="\n", strip=True)
    txt = ihtml.unescape(txt)
    # collapse repeated blank lines
    lines = [line.rstrip() for line in txt.splitlines()]
    return "\n".join([l for l in lines if l])

def load_framework_graph(
    uri, user, password,
    frameworks, control_areas, subcontrols,
    overviews, action_items, related_documents, additional_guidances, objective_criterias,
    database="neo4j",
    wipe=False
):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session(database="system") as sys_session:
        sys_session.run(f"CREATE DATABASE `{database}` IF NOT EXISTS WAIT")

    with driver.session(database=database) as session:
        # Frameworks + ControlAreas
        if wipe:
            session.run("MATCH (n) DETACH DELETE n")
        
        for f in frameworks:
            fid, fname, fdesc = f["id"], f["name"], f.get("description", "")
            for ca in [c for c in control_areas if c["framework_id"] == fid]:
                session.run("""
                MERGE (f:Framework {id: $fid})
                  ON CREATE SET f.name = $fname, f.description = $fdesc, f.createdAt=$ts
                MERGE (c:ControlArea {id: $cid})
                  ON CREATE SET c.name = $cname, c.description = $cdesc
                MERGE (f)-[:HAS_CONTROL_AREA]->(c)
                """, {
                    "fid": fid, "fname": fname, "fdesc": fdesc, "ts": datetime.utcnow().isoformat(),
                    "cid": ca["id"], "cname": ca["name"], "cdesc": ca.get("description", "")
                })

        # SubControls and children
        for sc in subcontrols:
            session.run("""
            MATCH (c:ControlArea {id:$cid})
            MERGE (s:SubControl {id:$sid})
              ON CREATE SET 
                        s.title=$title, 
                        s.description=$desc, 
                        s.notes=$notes, 
                        s.policy_section=$ps,
                        s.code=$identifier
            MERGE (c)-[:HAS_SUBCONTROL]->(s)
            """, {
                "cid": sc["control_id"],
                "sid": sc["id"], 
                "title": sc["title"], 
                "desc": sc.get("description", ""),
                "identifier": sc.get("identifier"),
                "notes": sc.get("notes", ""), 
                "ps": sc.get("policy_section", "")
            })

        # Overviews
        for o in overviews:
            session.run("""
            MATCH (s:SubControl {id:$sid})
            MERGE (o:Overview {id:$oid})
              ON CREATE SET o.description=$desc
            MERGE (s)-[:HAS_OVERVIEW]->(o)
            """, {"sid": o["subcontrol_id"], "oid": o["id"], "desc": o["description"]})

        # Action Items
        for a in action_items:
            session.run("""
            MATCH (s:SubControl {id:$sid})
            MERGE (a:ActionItem {id:$aid})
              ON CREATE SET a.description=$desc
            MERGE (s)-[:HAS_ACTION_ITEM]->(a)
            """, {"sid": a["subcontrol_id"], "aid": a["id"], "desc": a["description"]})
        
        # Objective Criteria
        for oc in objective_criterias:
            session.run("""
            MATCH (s:SubControl {id:$sid})
            MERGE (oc:ObjectiveCriteria {id:$ocid})
              ON CREATE SET oc.description=$desc
            MERGE (s)-[:HAS_OBJECTIVE_CRITERIA]->(oc)
            """, {"sid": oc["subcontrol_id"], "ocid": oc["id"], "desc": oc["description"]})

        # Related Documents
        related_docs = expand_related_documents(related_documents)
        for r in related_docs:
            session.run(
                """
                MATCH (s:SubControl {id: $sid})
                MERGE (d:EvidenceRequirement {id: $doc_id})
                ON CREATE SET
                    d.name        = $name,
                    d.first_seen  = datetime(),
                    d.example_raw = $raw
                ON MATCH SET
                    d.name        = coalesce(d.name, $name)
                MERGE (s)-[:REQUIRES_EVIDENCE]->(d)

                """,
                {
                    "sid": r["subcontrol_id"],
                    "doc_id": r["doc_id"],
                    "name": r["name"],
                    "raw": r["raw"],
                },
            )
            session.run(
                """
                MATCH (p:Policy)
                MATCH (d:EvidenceRequirement)
                WHERE
                (
                    p.norm_title IS NOT NULL
                    AND d.id IS NOT NULL
                    AND p.norm_title CONTAINS d.id
                )
                OR (
                    d.name IS NOT NULL
                    AND toLower(p.title) CONTAINS toLower(d.name)
                )
                MERGE (p)-[:SATISFIES_REQUIREMENT]->(d)
                """
            )
            
        # Additional Guidance
        for g in additional_guidances:
            session.run("""
            MATCH (s:SubControl {id:$sid})
            MERGE (g:AdditionalGuidance {id:$gid})
              ON CREATE SET g.description=$desc
            MERGE (s)-[:HAS_GUIDANCE]->(g)
            """, {"sid": g["subcontrol_id"], "gid": g["id"], "desc": g["description"]})

    driver.close()

def fetch_edges(uri, user, password, database="neo4j",
                start_framework=None,
                include_labels=("Framework","ControlArea","Subcontrol","Overview","ActionItem","RelatedDocument","AdditionalGuidance"),
                max_edges=1000):
    """
    Pull a thin edge list from Neo4j suitable for NetworkX.
    If start_framework is provided, we anchor the subgraph at that Framework node.
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    q_anchor = ""
    params = {"labels": list(include_labels), "max_edges": int(max_edges)}
    if start_framework:
        q_anchor = "MATCH (start:Framework) WHERE start.name = $fw OR start.title = $fw WITH start "
        params["fw"] = start_framework

    # Return light-weight triples (src, rel, dst) + human labels
    cypher = f"""
    {q_anchor}
    MATCH (n)-[r]->(m)
    { "WHERE (n)=start OR (start)-[*..3]->(n)" if start_framework else "" }
    AND ((labels(n)[0] IN $labels) OR (labels(m)[0] IN $labels))
    RETURN id(n) AS src_id,
           coalesce(n.title,n.name,n.identifier,toString(id(n))) AS src_text,
           labels(n) AS src_labels,
           type(r) AS rel,
           id(m) AS dst_id,
           coalesce(m.title,m.name,m.identifier,toString(id(m))) AS dst_text,
           labels(m) AS dst_labels
    LIMIT $max_edges
    """

    edges = []
    with driver.session(database=database) as s:
        for rec in s.run(cypher, **params):
            edges.append({
                "src_id": rec["src_id"],
                "src_text": rec["src_text"],
                "src_labels": rec["src_labels"],
                "rel": rec["rel"],
                "dst_id": rec["dst_id"],
                "dst_text": rec["dst_text"],
                "dst_labels": rec["dst_labels"],
            })
    driver.close()
    return edges

def build_nx_graph(edges):
    """
    Make a NetworkX MultiDiGraph with node attributes:
      - text (what to print)
      - labels (Neo4j labels)
    Edge attribute:
      - rel (relationship type)
    """
    G = nx.MultiDiGraph()
    for e in edges:
        if e["src_id"] not in G:
            G.add_node(e["src_id"], text=e["src_text"], labels=e["src_labels"])
        if e["dst_id"] not in G:
            G.add_node(e["dst_id"], text=e["dst_text"], labels=e["dst_labels"])
        G.add_edge(e["src_id"], e["dst_id"], rel=e["rel"])
    return G

def draw_graph(G, k=0.9, node_size=900, font_size=8):
    """
    Simple spring layout drawing.
    (No custom colors/styles to keep it lightweight and portable.)
    """
    pos = nx.spring_layout(G, k=k, seed=42)
    plt.figure(figsize=(12, 8))
    nx.draw_networkx_nodes(G, pos, node_size=node_size)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=10)
    # Node labels = node["text"]
    labels = {n: G.nodes[n].get("text", str(n)) for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=font_size)
    # Edge labels = relationship types
    edge_labels = {(u, v): d.get("rel","") for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=font_size)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [3]:
framework_map = {
    "SOC 2": "SOC 2",
    "NIST Cybersecurity Framework v2.0": "NIST CSF",
    "NIST Cybersecurity Framework v1.1": "NIST CSF",
    "NIST CSF v2.0": "NIST CSF",
    "NIST CSF 2.0": "NIST CSF",
    "NIST CSF2.0": "NIST CSF",
    "ISO 27001": "ISO 27001",
    "SOC2": "SOC 2",
    "SOC-2": "SOC 2",
    "SOC_2": "SOC 2",
    "NIST CSF": "NIST CSF",
}

In [49]:
def load_framework_from_json(json_file, db_name, wipe=False):
    with json_file.open("r", encoding="utf-8") as f:   
        frameworks = []
        
        data = json.load(f)
        extract_framework_data = extract_framework(data)
        framework_id = extract_framework_data[0]['id']
        framework_name = extract_framework_data[0]['name']
        framework_name = framework_map.get(framework_name, framework_name)
        framework_description = extract_framework_data[0]['description']
        frameworks.append({
            'id': framework_id,
            'name': framework_name,
            'description': framework_description
        })

        
        extract_apps_data = extract_apps(data)
        extract_sub_controls_data = extract_sub_controls(data)

        control_areas = []
        subcontrols = []

        overviews = []
        action_items = []
        related_documents = []
        additional_guidances = []
        objective_criterias = []

        for control_area in extract_apps_data:
            control_areas.append({
                'id': control_area.get('id'),
                'name': control_area.get('name'),
                'description': control_area.get('description'),
                'framework_id': framework_id
            })
        for subcontrol in extract_sub_controls_data:
            subcontrols.append({
                'id': subcontrol.get('id'),
                'title': subcontrol.get('title'),
                'description': subcontrol.get('description'),
                'notes': subcontrol.get('notes'),
                'policy_section': subcontrol.get('policy_section'),
                'control_id': subcontrol.get('app_id'),
                'identifier': subcontrol.get('identifier'),
            })

            formatted_data = convert_html_content_to_list(subcontrol['description'])
            overview =[f for f in formatted_data if f['title'] == 'Overview']
            action_item =[f for f in formatted_data if f['title'] == 'Action Items']
            related_document =[f for f in formatted_data if f['title'] == 'Related Documents']
            additional_guidance =[f for f in formatted_data if f['title'] == 'Additional Guidance']
            objective_criteria =[f for f in formatted_data if f['title'] == 'Objective Criteria']

            for index, ov in enumerate(overview, start=1):
                overviews.append({
                    'id': f"{subcontrol.get('id')}_OV{index}",
                    'subcontrol_id': subcontrol.get('id'),
                    'description': '\n'.join(ov.get('items'))
                })
            for i in objective_criteria:
                for index, item in enumerate(i.get('items'), start=1):
                    objective_criterias.append({
                        'id': f"{subcontrol.get('id')}_OC{index}",
                        'subcontrol_id': subcontrol.get('id'),
                        'description': item,
                    })
            for i in action_item:
                for index, item in enumerate(i.get('items'), start=1):
                    action_items.append({
                        'id': f"{subcontrol.get('id')}_AI{index}",
                        'subcontrol_id': subcontrol.get('id'),
                        'description': item,
                    })

            for i in related_document:
                for index, item in enumerate(i.get('items'), start=1):
                    related_documents.append({
                        'id': item,
                        'subcontrol_id': subcontrol.get('id'),
                        'description': item,
                    })
            for i in additional_guidance:
                for index, item in enumerate(i.get('items'), start=1):
                    additional_guidances.append({
                        'id': f"{subcontrol.get('id')}_AG{index}",
                        'subcontrol_id': subcontrol.get('id'),
                        'description': item,
                    })
        
        # load_framework_graph(
        #     NEO4J_URI,
        #     NEO4J_USER, NEO4J_PASS,
        #     frameworks, control_areas, subcontrols,
        #     overviews, action_items, related_documents, additional_guidances, objective_criterias,
        #     database=NEO4J_DATABASE,
        #     wipe=wipe

        # )

        import pandas as pd
        frameworks_df = pd.DataFrame(frameworks)
        control_areas_df = pd.DataFrame(control_areas)
        subcontrols_df = pd.DataFrame(subcontrols)
        overviews_df = pd.DataFrame(overviews)
        action_items_df = pd.DataFrame(action_items)
        related_documents_df = pd.DataFrame(related_documents)
        additional_guidances_df = pd.DataFrame(additional_guidances)
        objective_criterias_df = pd.DataFrame(objective_criterias)

        engine = create_engine(connection_url, echo=False)

        if wipe:
            with engine.connect() as conn:
                conn.execute(text("DROP TABLE IF EXISTS frameworks CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS control_areas CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS subcontrols CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS overviews CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS action_items CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS related_documents CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS additional_guidances CASCADE"))
                conn.execute(text("DROP TABLE IF EXISTS objective_criterias CASCADE"))
                conn.commit()
        if not frameworks_df.empty:
            frameworks_df.to_sql('frameworks', engine, if_exists='append', index=False)
        if not control_areas_df.empty:
            control_areas_df.to_sql('control_areas', engine, if_exists='append', index=False)
        if not subcontrols_df.empty:
            subcontrols_df.to_sql('subcontrols', engine, if_exists='append', index=False)
        if not overviews_df.empty:
            overviews_df.to_sql('overviews', engine, if_exists='append', index=False)
        if not action_items_df.empty:
            action_items_df.to_sql('action_items', engine, if_exists='append', index=False)
        if not related_documents_df.empty:
            related_documents_df.to_sql('related_documents', engine, if_exists='append', index=False)
        if not additional_guidances_df.empty:
            additional_guidances_df.to_sql('additional_guidances', engine, if_exists='append', index=False)
        if not objective_criterias_df.empty:
            objective_criterias_df.to_sql('objective_criterias', engine, if_exists='append', index=False)
        print(f"loaded {json_file.name} into database {DB_HOST}")
        

In [51]:

framework_dir = Path("../data/frameworks")
load_framework_from_json(Path("../data/frameworks/nist_cybersecurity_framework_v1_1.json"), 'policy-db', wipe=True)
# load_framework_from_json(Path("../data/frameworks/nist_cybersecurity_framework_v2_0.json"), 'policy-db', wipe=False)
load_framework_from_json(Path("../data/frameworks/SOC_2.json"), 'policy-db', wipe=False)
load_framework_from_json(Path("../data/frameworks/iso_27001_2022_framework.json"), 'policy-db', wipe=False)


loaded nist_cybersecurity_framework_v1_1.json into database <REDACTED>
loaded SOC_2.json into database <REDACTED>
loaded iso_27001_2022_framework.json into database <REDACTED>
